# moss-dna-gpt DemoThis notebook demonstrates the key features of moss-dna-gpt:1. **Model loading** — Load a trained DNA-GPT checkpoint2. **DNA completion** — Generate continuations from a DNA prefix3. **Variant effect scoring** — Score single-nucleotide variants4. **Markov baseline comparison** — Compare against classical baselines

## SetupInstall the package and import dependencies.

In [ ]:
import sysfrom pathlib import Path# Add project root to pathsys.path.insert(0, str(Path.cwd()))import torchfrom moss_dna_gpt.model import GPT, GPTConfigfrom moss_dna_gpt.sampling import samplefrom moss_dna_gpt.tokenizer import DnaTokenizerfrom moss_dna_gpt.scoring import score_variantfrom moss_dna_gpt.metrics import nats_to_bitsprint(f"PyTorch version: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")device = "cuda" if torch.cuda.is_available() else "cpu"print(f"Using device: {device}")

## Load a trained modelPoint to a checkpoint and load the model.

In [ ]:
CHECKPOINT = "repo/moss-dna-gpt-20m-patens/ckpt_step_8000000.pt"checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=True)model_cfg = GPTConfig(**checkpoint["model_config"])model = GPT(model_cfg).to(device)model.load_state_dict(checkpoint["model"])model.eval()tokenizer = DnaTokenizer()print(f"Model parameters: {model.num_parameters():,}")print(f"Context length: {model_cfg.block_size} bases")print(f"Training step: {checkpoint['step']:,}")

## DNA completionGiven a prefix, the model generates the most likely continuation base-by-base.

In [ ]:
prefix = "ACGTACGTACGT"max_new_tokens = 128temperature = 0.8top_k = 4prefix_ids = tokenizer.encode(prefix, unknown="n")idx = torch.tensor([prefix_ids], dtype=torch.long, device=device)allowed = [tokenizer.stoi[b] for b in tokenizer.dna_tokens]with torch.no_grad():    out = sample(model, idx, max_new_tokens=max_new_tokens,                 temperature=temperature, top_k=top_k,                 allowed_token_ids=allowed)full = tokenizer.decode(out[0].tolist(), skip_special=True)print(f"Prefix: {prefix}")print(f"Generated ({len(full)} bases):")print(full)

### Try different use cases- **AT-rich (promoter-like)**: `TATAAATAGCTAGCTAGCTAGC`- **GC-rich (coding-like)**: `GCCGCCATGGCCGAGCTCGAG`- **Simple repeat**: `ACGTACGTACGTACGTACGT`

In [ ]:
for label, seq in [    ("AT-rich region", "TATAAATAGCTAGCTAGCTAGC"),    ("GC-rich region", "GCCGCCATGGCCGAGCTCGAG"),    ("Simple repeat", "ACGTACGTACGTACGTACGT"),]:    ids = tokenizer.encode(seq, unknown="n")    idx = torch.tensor([ids], dtype=torch.long, device=device)    with torch.no_grad():        out = sample(model, idx, max_new_tokens=64,                     temperature=0.8, top_k=4, allowed_token_ids=allowed)    full = tokenizer.decode(out[0].tolist(), skip_special=True)    gc = (full.count("G") + full.count("C")) / len(full)    print(f"{label}: GC={gc:.1%} | {full[:80]}...")

## Variant Effect ScoringScore a single-nucleotide variant to see how surprising it is to the model.

In [ ]:
seq = "ACGTACGTACGTACGTACGTACGTACGT"pos = 5alt = "G"result = score_variant(model, tokenizer, seq, pos, alt, device=device)print(f"Reference: {seq}")print(f"Variant: position {pos} {seq[pos]} -> {alt}")print(f"  LLR: {result['llr']:+.4f} nats ({result['llr_bits']:+.4f} bits)")print(f"  Ref loss: {result['loss_ref']:.4f} nats/base")print(f"  Mut loss: {result['loss_mut']:.4f} nats/base")print(f"  Delta loss: {result['delta_loss']:+.6f} nats/base")print(f"  Interpretation: ", end="")if result["llr"] is not None and result["llr"] > 0:    print("Model prefers reference (mutation is surprising)")else:    print("Model tolerates or prefers the mutation")

## Markov Baseline ComparisonCompare the model's performance against classical Markov baselines.

In [ ]:
from moss_dna_gpt.markov import MarkovModeltrain_seqs = ["ACGTACGTACGTACGTACGT", "GCCGCCATGGCCGAGCTCGAG"]test_seqs = ["ACGTACGTACGT", "GCGCGCGCGCGC"]print("=== Markov Baselines (dummy data) ===")for order in [0, 1, 5]:    mm = MarkovModel(order, alpha=0.5).fit(train_seqs)    ce, n = mm.cross_entropy(test_seqs)    bits = nats_to_bits(ce)    print(f"  Markov-{order}: {bits:.4f} bits/base")print()print("For full evaluation on real data, run:")print("  python scripts/eval_markov.py --train-path <train> --test-path <test> --checkpoint <ckpt>")

## Next steps- Train your own model: `python scripts/run_real_quickstart.py --profile 5m --device auto`- Launch the Streamlit UI: `streamlit run apps/dna_chat.py`- Score variants from a CSV: `python scripts/score_variants.py`